# 📚 RAG Project — Notebook

Ce notebook implémente une application **RAG (Retrieval-Augmented Generation)** sur des documents PDF, TXT et DOCX.  
Le projet utilise **LangChain**, **ChromaDB**, et un **LLM HuggingFace** pour répondre aux questions à partir de documents de cours.

---

## 1️⃣ Install (exécuter une seule fois)

Installation des bibliothèques nécessaires : LangChain, ChromaDB, Transformers, PDF/DOCX handling, embeddings.

```python
!pip -q install -U \
  langchain langchain-community langchain-core langchain-text-splitters \
  langchain-huggingface chromadb \
  sentence-transformers transformers accelerate \
  pypdf python-docx

print("✅ Install OK")


In [35]:
# Install (exécuter une seule fois)
!pip -q install -U \
  langchain langchain-community langchain-core langchain-text-splitters \
  langchain-huggingface chromadb \
  sentence-transformers transformers accelerate \
  pypdf python-docx

print("✅ Install OK")

✅ Install OK


## 2️⃣ Configuration (⚙️ à modifier)

Définition des paramètres principaux : dossier de documents, chunking, modèle d’embeddings, LLM, paramètres de génération.

In [36]:
# ⚙️ 0) Configuration (modifie ici)

# Dossier qui contient tes documents
DATA_DIR = "/kaggle/input/mini-data"   # ex: "./data"

# Chunking
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

# Embeddings + Chroma
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
PERSIST_DIR = "./chroma_db_project2"
RETRIEVE_K = 5

# LLM (choisis un autre modèle ici)
# - "mistralai/Mistral-7B-Instruct-v0.2"   (plus lourd)
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# Génération
MAX_NEW_TOKENS = 300
DO_SAMPLE = False
TEMPERATURE = None  # ignoré si DO_SAMPLE=False


## 3️⃣ Imports et setup

Import des bibliothèques nécessaires et gestion des compatibilités entre différentes versions de LangChain.

In [37]:
import os
from typing import List, Optional

from pypdf import PdfReader
import docx

# LangChain imports (robustes selon versions)
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings HF
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except Exception:
    from langchain_community.embeddings import HuggingFaceEmbeddings

# Vectorstore Chroma (robuste selon versions)
try:
    from langchain_community.vectorstores import Chroma
except Exception:
    from langchain_chroma import Chroma

# LLM wrapper
try:
    from langchain_huggingface import HuggingFacePipeline
except Exception:
    from langchain_community.llms import HuggingFacePipeline

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("✅ Imports OK")

✅ Imports OK


## 4️⃣ Charger les documents (PDF / TXT / DOCX)

Fonction pour parcourir le dossier et extraire le texte des fichiers.
Chaque document est stocké avec ses métadonnées (source, filetype, page).

In [38]:
# 📁 1) Charger les documents (PDF / TXT / DOCX) -> List[Document]

def load_course_materials(path: str) -> List[Document]:
    docs: List[Document] = []
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dossier introuvable: {path}")

    for root, _, files in os.walk(path):
        for file in files:
            full = os.path.join(root, file)
            lower = file.lower()

            if lower.endswith(".pdf"):
                try:
                    reader = PdfReader(full)
                    for page_idx, page in enumerate(reader.pages):
                        text = (page.extract_text() or "").strip()
                        if not text:
                            continue
                        docs.append(Document(
                            page_content=text,
                            metadata={"source": full, "filetype": "pdf", "page": page_idx + 1},
                        ))
                except Exception as e:
                    print(f"⚠️ PDF erreur: {full} -> {e}")

            elif lower.endswith(".txt"):
                try:
                    with open(full, "r", encoding="utf-8", errors="ignore") as f:
                        text = f.read().strip()
                    if text:
                        docs.append(Document(
                            page_content=text,
                            metadata={"source": full, "filetype": "txt"},
                        ))
                except Exception as e:
                    print(f"⚠️ TXT erreur: {full} -> {e}")

            elif lower.endswith(".docx"):
                try:
                    d = docx.Document(full)
                    text = "\n".join([p.text for p in d.paragraphs]).strip()
                    if text:
                        docs.append(Document(
                            page_content=text,
                            metadata={"source": full, "filetype": "docx"},
                        ))
                except Exception as e:
                    print(f"⚠️ DOCX erreur: {full} -> {e}")

    return docs

docs_raw = load_course_materials(DATA_DIR)
print(f"✅ Documents chargés: {len(docs_raw)}")

if docs_raw:
    print("Exemple metadata:", docs_raw[0].metadata)
    print("Extrait:", docs_raw[0].page_content[:300])

✅ Documents chargés: 621
Exemple metadata: {'source': '/kaggle/input/mini-data/data/CH1_Big Data Analytics.pdf', 'filetype': 'pdf', 'page': 1}
Extrait: Master Big Data Analytics & Smart Systems (BDSaS)
Big Data Analytics II


## 5️⃣ Chunking

Découpe des documents en morceaux plus petits pour faciliter la recherche et la génération.

In [39]:
# ✂️ 2) Chunking

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

docs = splitter.split_documents(docs_raw)
print("✅ Chunks:", len(docs))

if docs:
    print("Exemple chunk metadata:", docs[0].metadata)
    print("Extrait chunk:", docs[0].page_content[:250])

✅ Chunks: 689
Exemple chunk metadata: {'source': '/kaggle/input/mini-data/data/CH1_Big Data Analytics.pdf', 'filetype': 'pdf', 'page': 1}
Extrait chunk: Master Big Data Analytics & Smart Systems (BDSaS)
Big Data Analytics II


## 6️⃣ Embeddings + ChromaDB

Création des embeddings avec HuggingFace et stockage dans ChromaDB.
Si la base existe déjà, elle est rechargée pour accélérer le processus.

In [40]:
# 🧠 3) Embeddings + ChromaDB (Vector DB)

embeddings = HuggingFaceEmbeddings(model_name=EMB_MODEL)

def get_or_build_vectordb(
    docs: List[Document],
    persist_dir: str,
):
    # Si déjà persisté, on recharge (plus rapide)
    if os.path.exists(persist_dir) and os.listdir(persist_dir):
        print("📦 Chroma existant détecté → chargement depuis", persist_dir)
        return Chroma(persist_directory=persist_dir, embedding_function=embeddings)

    print("🧱 Création Chroma depuis les documents…")
    vectordb = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=persist_dir,
    )
    try:
        vectordb.persist()
    except Exception:
        pass
    return vectordb

vectordb = get_or_build_vectordb(docs, PERSIST_DIR)
retriever = vectordb.as_retriever(search_kwargs={"k": RETRIEVE_K})

print("✅ ChromaDB prêt (persist_directory =", PERSIST_DIR, ")")

📦 Chroma existant détecté → chargement depuis ./chroma_db_project2
✅ ChromaDB prêt (persist_directory = ./chroma_db_project2 )


## 7️⃣ Charger le LLM HuggingFace

Fonction pour charger un modèle HF avec son tokenizer et pipeline.

In [41]:
# 🤖 4) Charger un autre modèle HF (facile à changer)

def load_hf_llm(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # certains modèles n'ont pas de pad_token → on le force pour éviter warnings/erreurs
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype="auto",
        trust_remote_code=True,
    )

    gen_pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        do_sample=DO_SAMPLE,
        temperature=TEMPERATURE if DO_SAMPLE else None,
        max_new_tokens=MAX_NEW_TOKENS,
        return_full_text=False,
    )

    llm = HuggingFacePipeline(pipeline=gen_pipe)
    return llm, tokenizer

llm, tokenizer = load_hf_llm(MODEL_NAME)
print("✅ LLM prêt:", MODEL_NAME)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Device set to use cuda:0


✅ LLM prêt: mistralai/Mistral-7B-Instruct-v0.2


## 8️⃣ RAG chain

Construction de la chaîne RAG pour générer des réponses à partir du contexte des documents

In [42]:
# 🧩 5) RAG chain (structure générique, compatible avec plusieurs modèles)

SYSTEM_MSG = """Tu es un assistant qui répond UNIQUEMENT à partir du contexte.

RÈGLES :
- Si la réponse n'est pas dans le contexte → réponds EXACTEMENT :
  "Je ne sais pas, ce n'est pas indiqué dans les documents."
- Ne fais AUCUNE déduction.
- Réponds toujours en français.
"""

def format_docs(docs: List[Document]) -> str:
    blocks = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", None)
        header = f"[SOURCE: {src}" + (f" | page {page}]" if page is not None else "]")
        blocks.append(header + "\n" + d.page_content)
    return "\n\n".join(blocks)

def build_chat_prompt(inputs: dict) -> str:
    context = inputs["context"]
    question = inputs["question"]

    user_msg = (
        f"Contexte :\n{context}\n\n"
        f"Question :\n{question}\n\n"
        f"Réponse :"
    )

    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": user_msg},
    ]

    # si le tokenizer a un chat template, on l'utilise (meilleur pour les modèles Instruct)
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    # fallback (rare) : prompt simple
    return SYSTEM_MSG + "\n\n" + user_msg

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(build_chat_prompt)
    | llm
    | StrOutputParser()
)

print("✅ RAG chain prête")

✅ RAG chain prête


## 9️⃣ Question -> réponse + sources

Fonction pour poser une question, afficher la réponse et les sources pertinentes.

In [43]:
# 🧪 6) Question -> réponse + sources

def ask(question: str, k: Optional[int] = None):
    k = k or RETRIEVE_K

    try:
        docs = retriever.get_relevant_documents(question)
    except Exception:
        docs = retriever.invoke(question)

    answer = rag_chain.invoke(question)

    print("QUESTION:", question)
    print("\nRÉPONSE:\n", answer)

    print("\nSOURCES (top-k):")
    for i, d in enumerate(docs[:k], 1):
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", "")
        if page:
            print(f"{i}. {src} (page {page})")
        else:
            print(f"{i}. {src}")

## 🔹 Exemples

In [44]:
# Exemples (adapte selon tes cours)
ask("Top-Down")
ask("un agent cognitif")
ask("un agent hybride")

QUESTION: Top-Down

RÉPONSE:
  Dans son approche, le Data Warehouse est un référentiel centralisé de l'entreprise stockant l'information au niveau le plus détaillé. Des Datamarts modélisés sous forme de schémas en étoile sont créés à partir de ce Data Warehouse. (Référence : page 25 du document MohamedMhaouach\_BI.pdf)

Top-p sampling (a.k.a., nucleus sampling) is proposed by sampling from the smallest set having a cumulative probability above (or equal to) p. (Référence : page 37 du document talk-fsdm-v4-posted\_Nfaoui.pdf)

Top-Down et Top-p sampling sont deux concepts différents. Le premier est une approche de conception de Data Warehouse, et le second est une méthode de échantillonnage utilisée dans le domaine de l'apprentissage automatique.

Je ne sais pas, ce n'est pas indiqué dans les documents comment ces concepts sont liés.

SOURCES (top-k):
1. /kaggle/input/mini-data/data/MohamedMhaouach_BI.pdf (page 25)
2. /kaggle/input/mini-data/data/talk-fsdm-v4-posted_Nfaoui.pdf (page 37)

In [45]:
ask("Classification hi´erarchique ascendante")

QUESTION: Classification hi´erarchique ascendante

RÉPONSE:
  Classification hiérarchique ascendante est une méthode de classification non supervisée. Elle utilise des critères d'agrégation explicites, tels que le critère du saut minimal (Single linkage clustering), le critère du saut maximal (Complete linkage clustering), et le critère de la moyenne (Average linkage clustering), pour déterminer la distance entre deux classes contenant plus d'un individu. Ce processus continue jusqu'à ce que toutes les classes soient regroupées en une seule classe ou qu'il n'y ait plus de division possible.

Cette méthode a pour objectif de réduire le nombre de classes en les regroupant en fonction de leur similitude. Elle est utilisée dans la recherche d'information, la réduction de documents, et dans d'autres applications de la fouille de textes.

Cependant, il n'est pas indiqué dans le contexte si ce modèle en flocon (clustering) est utilisé dans le cadre de la classification hiérarchique ascendante

In [46]:
ask("SVM")

QUESTION: SVM

RÉPONSE:
  Je ne sais pas, ce n'est pas indiqué dans les documents.

SOURCES (top-k):
1. /kaggle/input/mini-data/data/4 CF model-based Matrix factorization vers 1.pdf (page 2)
2. /kaggle/input/mini-data/data/3 Memory-based Collaborative Filtering-3-1-2 (1).pdf (page 2)
3. /kaggle/input/mini-data/data/4 CF model-based Matrix factorization vers 1.pdf (page 10)
4. /kaggle/input/mini-data/data/talk-fsdm-v4-posted_Nfaoui.pdf (page 8)
5. /kaggle/input/mini-data/data/talk-fsdm-v4-posted_Nfaoui.pdf (page 81)
